# 21장 seq2seq와 어텐션 — 어디를 얼마나 볼 것인가

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch21_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `p01_attention.py` 를 치고 실행. 첫 행을 종이로 계산해 대조 | [ch21_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch21_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | 세 단계(점수·가중치·가중 평균)의 shape을 각 줄에 | [ch21_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch21_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 복원. 그다음 질의·키·값 선형층을 추가한 `Attention` 클래스로 만들기(22장 `MultiHeadAttention` 의 헤드 1개 판) | [ch21_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch21_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "21"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/21")

준비 끝. 작업 폴더: code/21


## `p01_attention.py` — 21장 어텐션의 계산을 가장 작은 예로: 질의(query) 하나가 키(key) 여러 개 중 어디를 얼마나 볼지

첫 행을 종이로 계산해 대조하세요.

In [ ]:
# 21장 어텐션의 계산을 가장 작은 예로: 질의(query) 하나가 키(key) 여러 개 중 어디를 얼마나 볼지
import torch
import torch.nn.functional as F

torch.manual_seed(0)

In [ ]:
# 문장 하나: 토큰 5개, 각 토큰은 4차원 벡터 (19장의 임베딩 결과라고 생각)
tokens = ["원고는", "피고에게", "손해를", "배상하라고", "청구하였다"]
x = torch.randn(5, 4)

In [ ]:
# 1) 점수: 질의와 각 키의 내적. 여기서는 질의 = 키 = 값 = x (self-attention)
q = x[3]                                    # "배상하라고" 의 관점에서
scores = x @ q                              # [5]: 다른 토큰들과 얼마나 맞는가
print("점수      :", [round(s, 2) for s in scores.tolist()])

점수      : [-3.26, -3.52, 1.54, 6.05, 5.14]


In [ ]:
# 2) 가중치: softmax 로 합이 1 인 확률로
w = F.softmax(scores / x.shape[1] ** 0.5, dim=0)   # sqrt(d) 로 나누는 것은 값이 너무 커지는 것을 막기 위해
print("가중치    :", {t: round(v, 2) for t, v in zip(tokens, w.tolist())}, "| 합", round(w.sum().item(), 2))

가중치    : {'원고는': 0.01, '피고에게': 0.0, '손해를': 0.06, '배상하라고': 0.57, '청구하였다': 0.36} | 합 1.0


In [ ]:
# 3) 출력: 값(value) 들의 가중 평균
out = w @ x                                 # [4]
print("출력 shape:", tuple(out.shape), "= 가중치로 섞은 벡터")

출력 shape: (4,) = 가중치로 섞은 벡터


In [ ]:
# 모든 토큰이 동시에 질의가 되면 → 행렬 하나로 끝난다
W = F.softmax((x @ x.T) / x.shape[1] ** 0.5, dim=1)     # [5, 5]: i 행 = i 번 토큰이 각 토큰을 보는 가중치
print("\n어텐션 행렬 [5,5] (행 합 = 1):")
for t, row in zip(tokens, W.tolist()):
    print(f"  {t:<6}", [round(v, 2) for v in row])
print("출력:", tuple((W @ x).shape), "= 입력과 같은 [5, 4], 각 토큰이 문맥을 섞어 넣은 새 벡터")


어텐션 행렬 [5,5] (행 합 = 1):
  원고는    [0.64, 0.19, 0.1, 0.03, 0.03]
  피고에게   [0.05, 0.89, 0.05, 0.01, 0.01]
  손해를    [0.05, 0.11, 0.27, 0.18, 0.39]
  배상하라고  [0.01, 0.0, 0.06, 0.57, 0.36]
  청구하였다  [0.01, 0.01, 0.11, 0.3, 0.58]
출력: (5, 4) = 입력과 같은 [5, 4], 각 토큰이 문맥을 섞어 넣은 새 벡터


In [ ]:
# 19장의 '평균' 과 비교: 평균은 가중치가 전부 1/5 인 어텐션이다
print("\n단어 주머니(평균)의 가중치:", [round(1 / 5, 2)] * 5)


단어 주머니(평균)의 가중치: [0.2, 0.2, 0.2, 0.2, 0.2]


## 다음 단계

- 2단계 「주석 달기」: [ch21_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch21_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch21_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch21_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.